In [2]:
import spatial_sdm as sdm

In [2]:
# ============================================================
# Spatial Durbin Model (SDM) – Main Execution Script
# ============================================================

# spatial_sdm.py
"""
Spatial SDM (Spatial Durbin Model) for panel data in PyMC.
...
"""
from __future__ import annotations
from pathlib import Path

import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import os


# ------------------------------------------------------------
# 0) Define Data Path
# ------------------------------------------------------------
try:
    # 1. Preferred method for scripts (.py files):
    # Get the directory of the current script file.
    BASE_DIR = Path(__file__).parent
except NameError:
    # 2. Fallback for notebooks (Colab/Jupyter):
    # Use the current working directory, which should be the Git root
    # after cloning and changing directory (see Colab setup below).
    BASE_DIR = Path(os.getcwd())

# Define the relative path to the raw data folder
RAW_DATA_PATH = BASE_DIR / "data" / "raw"

# Sanity check (Optional, but useful for debugging)
if not RAW_DATA_PATH.is_dir():
    raise FileNotFoundError(
        f"The data/raw directory was not found at the expected path: {RAW_DATA_PATH}"
    )

print(f"Data will be loaded from: {RAW_DATA_PATH}")

# ------------------------------------------------------------
# 1) Load data
# ------------------------------------------------------------
# ------------------------------------------------------------
# Use the RAW_DATA_PATH variable and the / operator from pathlib
# to construct the full file path.

df_sorted = pd.read_excel(RAW_DATA_PATH / "df_sorted.xlsx")
W_cul04_raw = pd.read_excel(RAW_DATA_PATH / "W_cul04.xlsx")
W_cul06_raw = pd.read_excel(RAW_DATA_PATH / "W_cul06.xlsx")
W_cul05_raw = pd.read_excel(RAW_DATA_PATH / "W_cul05.xlsx")
W_geo_raw = pd.read_excel(RAW_DATA_PATH / "W_geo.xlsx")
W_trade_raw = pd.read_excel(RAW_DATA_PATH / "W_trade.xlsx")


# ------------------------------------------------------------
# 2) Clean spatial weights and align with panel data
# ------------------------------------------------------------
W_cul04 = sdm.prepare_W_from_excel(W_cul04_raw)
W_cul06 = sdm.prepare_W_from_excel(W_cul06_raw)
W_cul05 = sdm.prepare_W_from_excel(W_cul05_raw)
W_geo = sdm.prepare_W_from_excel(W_geo_raw)
W_trade = sdm.prepare_W_from_excel(W_trade_raw)


df, W = sdm.align_df_and_W(df_sorted, W_cul04)

# Ensure correct sorting (VERY IMPORTANT)
df = df.sort_values(["year", "country"]).copy()
df["year"] = df["year"].astype(int)

# ------------------------------------------------------------
# 3) Sanity checks
# ------------------------------------------------------------
years = sorted(df["year"].unique())
countries = sorted(df["country"].unique())

print("Number of countries (N):", len(countries))
print("Number of years (T):", len(years))
print("Number of observations (NT):", df.shape[0])
print("Expected NT = N * T:", len(countries) * len(years))
print("Balanced panel:", df.shape[0] == len(countries) * len(years))
print("W shape:", W.shape)

# ------------------------------------------------------------
# 4) Fit SDM on the full dataset
# ------------------------------------------------------------



trace_full, countries_sorted, years_sorted, W_base = sdm.run_sdm_model(
    df, W,
    draws=1000,
    tune=1000,
    chains=4,
    target_accept=0.95,
    random_seed=123,
    cores=None,
    progressbar=True,
)

print("SDM estimation completed.")

# ------------------------------------------------------------
# 5) (Optional) Save posterior samples for reproducibility
# ------------------------------------------------------------
import arviz as az
az.to_netcdf(trace_full, "trace_sdm_full.nc")



FileNotFoundError: The data/raw directory was not found at the expected path: /content/data/raw

In [ ]:
import importlib
import spatial_sdm as sdm
importlib.reload(sdm)

summary_main, df_alpha, df_time_alpha = sdm.summarize_sdm_trace(
    trace_full, countries_sorted, years_sorted, round_to=4, verbose=True
)

summary_main
df_alpha.head()
df_time_alpha.head()


Main coefficients summary:
             mean      sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd   ess_bulk  \
rho      -0.7651  0.1244 -0.9480  -0.5443     0.0015   0.0013  6506.9734   
beta[0]   0.2751  0.1578 -0.0324   0.5571     0.0019   0.0017  7011.6061   
beta[1]   0.5150  0.1132  0.3051   0.7290     0.0015   0.0012  5465.6532   
beta[2]   0.3356  0.1099  0.1371   0.5459     0.0017   0.0012  4300.1137   
beta[3]   0.4604  0.1745  0.1416   0.7947     0.0024   0.0018  5352.1950   
gamma[0]  2.1360  0.3133  1.5702   2.7469     0.0045   0.0035  4930.8249   
gamma[1]  0.0818  0.4461 -0.7752   0.9081     0.0063   0.0047  4986.0477   
gamma[2]  0.9558  0.3940  0.2528   1.7323     0.0053   0.0043  5541.7387   
gamma[3]  0.2240  0.6169 -0.9254   1.3774     0.0093   0.0070  4448.2800   
sigma     0.3067  0.0178  0.2741   0.3412     0.0002   0.0002  6842.7035   

           ess_tail   r_hat  
rho       5188.0777  1.0003  
beta[0]   5383.0062  0.9999  
beta[1]   5778.5365  1.0009  
beta[2]   5309.

,year,time_alpha_mean,time_alpha_sd
0,2002,-0.483513,0.478709
1,2003,-0.448861,0.458866
2,2004,-0.136034,0.460609
3,2005,-0.220091,0.464483
4,2006,0.102899,0.477259


In [ ]:
import numpy as np
import pandas as pd
import arviz as az
import spatial_sdm as sdm

# ------------------------------------------------------------
# Helper: build X, WX, indices (must match model ordering)
# ------------------------------------------------------------
def build_design_mats(df_sorted, W_base):
    """
    Build X, WX, y, country_idx, year_idx consistent with run_sdm_model ordering.

    Assumes df_sorted is sorted by ['year','country'] and panel is balanced.
    """
    df_sorted = df_sorted.sort_values(["year", "country"]).copy()
    df_sorted["year"] = df_sorted["year"].astype(int)

    years_sorted = sorted(df_sorted["year"].unique())
    countries_sorted = sorted(df_sorted["country"].unique())
    T, N = len(years_sorted), len(countries_sorted)

    country_to_idx = {c: i for i, c in enumerate(countries_sorted)}
    year_to_idx = {y: i for i, y in enumerate(years_sorted)}

    country_idx = df_sorted["country"].map(country_to_idx).to_numpy("int32")
    year_idx = df_sorted["year"].map(year_to_idx).to_numpy("int32")

    X = df_sorted[['GDP','Political Stability','exchange rate',"Rule of Law: Estimate"]].to_numpy("float64")
    y = df_sorted["inbound"].to_numpy("float64")

    NT, K = X.shape
    assert NT == N * T, "Balanced panel required (NT == N*T)."

    WX = np.zeros((NT, K), dtype="float64")
    for t in range(T):
        X_t = X[t*N:(t+1)*N, :]
        WX[t*N:(t+1)*N, :] = W_base @ X_t

    return X, WX, y, country_idx, year_idx, countries_sorted, years_sorted, N, T


# ------------------------------------------------------------
# 1) Build matrices for prediction diagnostics
# ------------------------------------------------------------
X, WX, y, country_idx, year_idx, countries_sorted, years_sorted, N, T = build_design_mats(df, W_base)

# ------------------------------------------------------------
# 2) Posterior mean prediction and MSE by country
# ------------------------------------------------------------
# If you still have your compute_sdm_predictions() function in the notebook, use it.
# Otherwise, here is a compact version:

def posterior_mean_predictions(trace, X, WX, country_idx, year_idx, W_base):
    """
    Compute posterior mean of E[y | params] for each observation:
        y_hat_t = (I - rho W)^(-1) * XB_t
    """
    post = trace.posterior
    beta = post["beta"].mean(dim=("chain","draw")).values
    gamma = post["gamma"].mean(dim=("chain","draw")).values
    alpha = post["alpha"].mean(dim=("chain","draw")).values
    time_alpha = post["time_alpha"].mean(dim=("chain","draw")).values
    rho = float(post["rho"].mean(dim=("chain","draw")).values)

    XB = X @ beta + WX @ gamma + alpha[country_idx] + time_alpha[year_idx]

    S = np.linalg.inv(np.eye(W_base.shape[0]) - rho * W_base)

    y_hat = np.zeros_like(XB)
    for t in range(T):
        y_hat[t*N:(t+1)*N] = S @ XB[t*N:(t+1)*N]

    return y_hat

y_hat = posterior_mean_predictions(trace_full, X, WX, country_idx, year_idx, W_base)

mse_by_country = {}
for c_i, c in enumerate(countries_sorted):
    mask = (country_idx == c_i)
    mse_by_country[c] = float(np.mean((y[mask] - y_hat[mask])**2))

mse_df = pd.DataFrame({"country": list(mse_by_country.keys()),
                       "mse": list(mse_by_country.values())}).sort_values("mse")

print("MSE by country (lower is better):")
display(mse_df)

# Optional: save
mse_df.to_csv("mse_by_country.csv", index=False)


# ------------------------------------------------------------
# 3) Moran's I on SDM residuals by year
# ------------------------------------------------------------
# We need a PySAL weights object + residuals per year
from libpysal.weights import W as W_pysal
from esda.moran import Moran

# Build PySAL W object from W_base
neighbors = {i: list(np.where(W_base[i] > 0)[0]) for i in range(N)}
weights = {i: W_base[i, neighbors[i]] for i in range(N)}
W_obj = W_pysal(neighbors, weights)

# Extract posterior means
post = trace_full.posterior
rho_mean = float(post["rho"].mean(dim=("chain","draw")).values)
beta_mean = post["beta"].mean(dim=("chain","draw")).values
gamma_mean = post["gamma"].mean(dim=("chain","draw")).values
alpha_mean = post["alpha"].mean(dim=("chain","draw")).values
time_alpha_mean = post["time_alpha"].mean(dim=("chain","draw")).values

# Linear component XB
XB = (X @ beta_mean) + (WX @ gamma_mean) + alpha_mean[country_idx] + time_alpha_mean[year_idx]

# SDM "structural" residuals per year:
# y - rho*W y - XB
resid = np.zeros_like(y)
for t in range(T):
    y_t = y[t*N:(t+1)*N]
    XB_t = XB[t*N:(t+1)*N]
    resid[t*N:(t+1)*N] = y_t - rho_mean * (W_base @ y_t) - XB_t

# Moran's I by year
morans_rows = []
for t in range(T):
    r_t = resid[t*N:(t+1)*N]
    mi = Moran(r_t, W_obj)
    morans_rows.append({
        "year": int(years_sorted[t]),
        "moran_I": float(mi.I),
        "p_norm": float(mi.p_norm),      # normal approximation p-value
        "p_sim": float(mi.p_sim),        # permutation p-value (default perms in esda)
    })

morans_df = pd.DataFrame(morans_rows)
print("Moran's I of residuals by year:")
display(morans_df)

# Optional: save
morans_df.to_csv("moransI_residuals_by_year.csv", index=False)


MSE by country (lower is better):


,country,mse
10,Türkiye,0.025088
2,Egypt,0.039710
5,Iran,0.047048
4,India,0.048984
9,Saudi Arabia,0.063582
6,Italy,0.074127
0,Azerbaijan,0.079209
1,China,0.097253
7,Kazakhstan,0.110343
3,Georgia,0.223492


Moran's I of residuals by year:


,year,moran_I,p_norm,p_sim
0,2002,-0.223483,0.243545,0.061
1,2003,-0.163656,0.547729,0.280
2,2004,-0.182952,0.433397,0.192
3,2005,-0.222078,0.248953,0.123
4,2006,-0.171907,0.497086,0.279
5,2007,-0.006276,0.376089,0.179
6,2008,0.006884,0.312779,0.157
7,2009,-0.185051,0.421848,0.100
8,2010,-0.133088,0.754675,0.371
9,2011,-0.139841,0.706727,0.333


In [ ]:
# ------------------------------------------------------------
# Leave-One-Country-Out (LOCO) cross-validation
# ------------------------------------------------------------
# NOTE: This is computationally expensive.
# Start with small draws/tune for testing.

res_loco = sdm.loco_cv(
    df=df,
    W_df=W,
    draws=200,           # increase after testing
    tune=200,
    chains=2,
    target_accept=0.95,
    random_seed=123,
    cores=None,
    progressbar=False,
)

print(res_loco["status"].value_counts())
print(res_loco.head())

# Save LOCO results
res_loco.to_csv("loco_results.csv", index=False)

print("LOCO cross-validation completed.")
